In [ ]:
!wget -O MMM_MMM_Comptage.csv \
  "https://data.montpellier3m.fr/sites/default/files/ressources/MMM_MMM_Comptage.csv"

--2026-08-10 11:37:01--  https://data.montpellier3m.fr/sites/default/files/ressources/MMM_MMM_Comptage.csv
Resolving data.montpellier3m.fr (data.montpellier3m.fr)... 193.227.228.225
Connecting to data.montpellier3m.fr (data.montpellier3m.fr)|193.227.228.225|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3871372 (3.7M) [text/csv]
Saving to: ‘MMM_MMM_Comptage.csv’

MMM_MMM_Comptage.cs 100%[===================>]   3.69M  3.24MB/s    in 1.1s    

2026-08-10 11:37:03 (3.24 MB/s) - ‘MMM_MMM_Comptage.csv’ saved [3871372/3871372]



In [ ]:
!curl -sL "https://data.montpellier3m.fr/dataset/comptages-horaires-velo-et-pieton-issus-des-compteurs-de-velos" -o page.html

In [ ]:
!grep -oE 'https://data\.montpellier3m\.fr/sites/default/files/ressources/[^"]+\.csv' page.html

In [ ]:
# Шаг 1 — посмотри, что вообще curl получил (может быть просто пустая заглушка или редирект)
!wc -l page.html
!grep -c "csv" page.html
!grep -i "csv" page.html | head -5

366 page.html
1
var _paq = _paq || [];(function(){var u=(("https:" == document.location.protocol) ? "https://matomo.montpellier.fr/" : "https://matomo.montpellier.fr/");_paq.push(["setSiteId", "31"]);_paq.push(["setTrackerUrl", u+"matomo.php"]);_paq.push(["setDownloadExtensions", "aac|arc|arj|asf|asx|avi|bin|csv|doc(x|m)?|dot(x|m)?|exe|flv|gif|gz|gzip|hqx|jar|jpe?g|js|mp(2|3|4|e?g)|mov(ie)?|msi|msp|pdf|phps|png|ppt(x|m)?|pot(x|m)?|pps(x|m)?|ppam|sld(x|m)?|thmx|qtm?|ra(m|r)?|sea|sit|tar|tgz|torrent|txt|wav|wma|wmv|wpd|xls(x|m|b)?|xlt(x|m)|xlam|xml|z|zip|7z|bin|csv|doc|dwg|geojson|gpx|gtfs|json|jsonld|kml|kmz|ods|pdf|shp|txt|xls|xlsx|xml|xsd|zip|docx"]);_paq.push(["setDoNotTrack", 1]);_paq.push(["trackPageView"]);_paq.push(["setIgnoreClasses", ["no-tracking","colorbox"]]);_paq.push(["enableLinkTracking"]);var d=document,g=d.createElement("script"),s=d.getElementsByTagName("script")[0];g.type="text/javascript";g.defer=true;g.async=true;g.src=u+"matomo.js";s.parentNode.insertBefore(g,s);})

In [ ]:
# Шаг 2 — более широкий поиск на случай, если ссылка относительная или с другим доменом
!grep -oE 'href="[^"]*\.csv[^"]*"' page.html | head -10
!grep -oE '"downloadURL"\s*:\s*"[^"]+"' page.html | head -10

In [ ]:
# Шаг 3 — параллельно пробуем DKAN API напрямую по метаданным датасета (обходит проблему JS-рендеринга)
!curl -s "https://data.montpellier3m.fr/api/1/metastore/schemas/dataset/items/comptages-horaires-velo-et-pieton-issus-des-compteurs-de-velos" \
  -o dataset_meta.json
!cat dataset_meta.json | python3 -m json.tool | head -50

Expecting value: line 1 column 1 (char 0)


In [ ]:
# Шаг 4 — находим правильный slug через поиск по каталогу датасетов
!curl -s "https://data.montpellier3m.fr/api/1/search?fulltext=velo" -o search.json
!cat search.json | python3 -m json.tool | grep -i -E '"identifier"|"title"' | head -30

Expecting value: line 1 column 1 (char 0)


In [ ]:
!wget -O geoloc_compteurs.csv \
  "https://data.montpellier3m.fr/sites/default/files/ressources/MMM_MMM_GeolocCompteurs.csv"

--2026-08-11 06:31:31--  https://data.montpellier3m.fr/sites/default/files/ressources/MMM_MMM_GeolocCompteurs.csv
Resolving data.montpellier3m.fr (data.montpellier3m.fr)... 193.227.228.225
Connecting to data.montpellier3m.fr (data.montpellier3m.fr)|193.227.228.225|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4488 (4.4K) [text/csv]
Saving to: ‘geoloc_compteurs.csv’

geoloc_compteurs.cs 100%[===================>]   4.38K  --.-KB/s    in 0.001s  

2026-08-11 06:31:31 (4.52 MB/s) - ‘geoloc_compteurs.csv’ saved [4488/4488]



In [ ]:
# 2. Смотрим на структуру, чтобы понять, в какой колонке серийный номер
import pandas as pd
geoloc = pd.read_csv("geoloc_compteurs.csv", sep=',')  # или без sep — зависит от факта, проверь как в задаче 5
print(geoloc.columns.tolist())

['Nom du com', 'N° Série', 'Latitude', 'Longitude', 'OSM_Line_i', 'Ancien N° Série']


In [ ]:
geoloc

,Nom du com,N° Série,Latitude,Longitude,OSM_Line_i,Ancien N° Série
0,Compteur Vélo Tanneurs,XTH19101158,43.616209,3.874408,188609530,NaN
1,Compteur Piéton/Vélo Berracasa,X2H19070220,43.609699,3.896940,121403593,NaN
2,Compteur Vélo Lodève Celleneuve,Y2H20042633,43.614650,3.833600,734202564,NaN
3,Compteur Vélo Lavérune,X2H24042101,43.590700,3.813240,97705885,NaN
4,Compteur Vélo Vieille poste,ZLT25011699,43.615742,3.909632,676645909,NaN
5,Compteur Vélo Delmas 2,Y2H20063164,43.626698,3.895629,105575465,Y2H20063164
6,Compteur Vélo Delmas 1,ZLT26063736,43.626698,3.895629,105575465,Y2H20063163
7,Compteur Vélo Gerhardt 1,Y2H20063162,43.613884,3.868467,23231541,NaN
8,Compteur Vélo Lattes 2,Y2H20042634,43.579260,3.933270,25871951,NaN
9,Compteur Vélo Lattes 1,Y2H20042635,43.578830,3.933240,137058167,NaN


In [ ]:
# 3. Скачиваем архив (историю) для каждого счётчика из списка
import time

# замени 'serial_column' на реальное имя колонки с серийным номером после шага 2
serials = geoloc['N° Série'].tolist()

import os
os.makedirs("eco_archives", exist_ok=True)

for s in serials:
    url = f"https://data.montpellier3m.fr/sites/default/files/ressources/MMM_EcoCompt_{s}_archive.json"
    out = f"eco_archives/MMM_EcoCompt_{s}_archive.json"
    !wget -q -O "{out}" "{url}"
    print(s, "->", os.path.getsize(out) if os.path.exists(out) else "FAILED")
    time.sleep(0.5)  # не долбить сервер слишком часто

XTH19101158 -> 604613
X2H19070220 -> 603547
Y2H20042633 -> 0
X2H24042101 -> 28921
ZLT25011699 -> 113540
Y2H20063164 -> 0
ZLT26063736 -> 13277
Y2H20063162 -> 0
Y2H20042634 -> 0
Y2H20042635 -> 0
X2H20104132 -> 572717
X2H21070341 -> 458229
ZLT26063735 -> 10628
X2H21070343 -> 470153
X2H21070347 -> 423356
X2H21070344 -> 647864
X2H21070345 -> 472266
X2H21070346 -> 473039
X2H21070348 -> 584483
X2H21070349 -> 487657
X2H25023006 -> 38639
ZLT26043541 -> 10651
XTH21015106 -> 466457
XTH24072390 -> 382183
ZLT26063734 -> 10732
X2H21111121 -> 463635
X2H22043029 -> 425399
X2H22043030 -> 438256
X2H22043031 -> 439660
X2H22043032 -> 439540
X2H22043033 -> 409500
X2H22043034 -> 396387
X2H22043035 -> 425492
X2H22104770 -> 478331
X2H22104768 -> 367311
X2H22104775 -> 370317
X2H22104776 -> 367429
X2H22104773 -> 359335
X2H22104774 -> 370402
X2H22104767 -> 365758
X2H22104766 -> 367071
X2H22104765 -> 366900
X2H22104769 -> 481044
X2H22104777 -> 384408
ZLT26063738 -> 12393
X2H22104771 -> 324798
COM23120110 -> 11454

In [ ]:
import shutil
from google.colab import files

In [ ]:
# Создается архив
shutil.make_archive("eco_archives", 'zip', "eco_archives")

# Скачивание архива на ваш компьютер
files.download(f'{"eco_archives"}.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import requests
import json

BASE = "https://portail-api.montpellier3m.fr"

# NGSI-LD почти всегда требует либо заголовок Link с @context,
# либо Content-Type: application/ld+json — пробуем оба варианта ниже.
HEADERS_LD = {
    "Accept": "application/ld+json",
}
HEADERS_LD_CONTEXT = {
    "Accept": "application/json",
    "Link": '<https://uri.etsi.org/ngsi-ld/v1/ngsi-ld-core-context.jsonld>; '
            'rel="http://www.w3.org/ns/json-ld#context"; type="application/ld+json"',
}

def try_get(path, headers, params=None, label=""):
    url = f"{BASE}{path}"
    print(f"\n=== {label or path} ===")
    print("URL:", url, "params:", params)
    try:
        r = requests.get(url, headers=headers, params=params, timeout=15)
        print("Status:", r.status_code)
        print("Response headers (обрати внимание на NGSILD-Results-Count, если есть):")
        for k in ("NGSILD-Results-Count", "Link", "www-authenticate"):
            if k in r.headers:
                print(f"  {k}: {r.headers[k]}")
        # печатаем начало тела, а не всё — чтобы не заваливать вывод
        text = r.text[:1500]
        print("Тело ответа (первые 1500 симв.):")
        print(text)
        return r
    except Exception as e:
        print("Ошибка запроса:", e)
        return None

# --- Шаг 1: список доступных типов сущностей (стандартный NGSI-LD endpoint) ---
r1 = try_get("/ngsi-ld/v1/types", HEADERS_LD, label="Список типов сущностей")

# --- Шаг 2: если типы не отдались напрямую, пробуем entities без фильтра по типу ---
r2 = try_get("/ngsi-ld/v1/entities",
             HEADERS_LD_CONTEXT,
             params={"limit": 5},
             label="Первые 5 любых сущностей (без фильтра типа)")

# --- Шаг 3: пробуем стандартный Fiware Smart Data Model для транспортного потока ---
# TrafficFlowObserved — стандартный тип из смарт-моделей Fiware для счётчиков потока;
# это ПРЕДПОЛОЖЕНИЕ, не подтверждённый факт для конкретно этого портала.
for etype in ["TrafficFlowObserved", "BikeHireDockingStation", "Device", "PedestrianFlowObserved"]:
    try_get("/ngsi-ld/v1/entities",
            HEADERS_LD_CONTEXT,
            params={"type": etype, "limit": 3},
            label=f"Сущности типа {etype}")

# --- Шаг 4: если знаем конкретный серийник, пробуем угадать id по конвенции urn ---
# Это тоже предположение по конвенции NGSI-LD (urn:ngsi-ld:<Type>:<id>) — подставь
# серийник, который точно есть в geoloc-файле, для проверки.
test_serial = "XTH19101158"
for etype in ["TrafficFlowObserved", "Device"]:
    entity_id = f"urn:ngsi-ld:{etype}:MMM_EcoCompt_{test_serial}"
    try_get(f"/ngsi-ld/v1/entities/{entity_id}",
            HEADERS_LD_CONTEXT,
            label=f"Прямой запрос сущности {entity_id}")


=== Список типов сущностей ===
URL: https://portail-api.montpellier3m.fr/ngsi-ld/v1/types params: None
Status: 404
Response headers (обрати внимание на NGSILD-Results-Count, если есть):
Тело ответа (первые 1500 симв.):
<html>
<head><title>404 Not Found</title></head>
<body>
<center><h1>404 Not Found</h1></center>
<hr><center>nginx</center>
</body>
</html>


=== Первые 5 любых сущностей (без фильтра типа) ===
URL: https://portail-api.montpellier3m.fr/ngsi-ld/v1/entities params: {'limit': 5}
Status: 404
Response headers (обрати внимание на NGSILD-Results-Count, если есть):
Тело ответа (первые 1500 симв.):
<html>
<head><title>404 Not Found</title></head>
<body>
<center><h1>404 Not Found</h1></center>
<hr><center>nginx</center>
</body>
</html>


=== Сущности типа TrafficFlowObserved ===
URL: https://portail-api.montpellier3m.fr/ngsi-ld/v1/entities params: {'type': 'TrafficFlowObserved', 'limit': 3}
Status: 404
Response headers (обрати внимание на NGSILD-Results-Count, если есть):
Тело отв

In [ ]:
import requests
import json

BASE = "https://portail-api-data.montpellier3m.fr"

def try_get(path, params=None, headers=None, label=""):
    url = f"{BASE}{path}"
    print(f"\n=== {label or path} ===")
    print("URL:", url, "params:", params)
    try:
        r = requests.get(url, params=params, headers=headers or {}, timeout=20)
        print("Status:", r.status_code)
        print("Тело (первые 1500 симв.):")
        print(r.text[:1500])
        return r
    except Exception as e:
        print("Ошибка:", e)
        return None

# --- Шаг 1: список счётчиков — узнаём точный формат ecocounterId ---
r1 = try_get("/ecocounter", label="Список эко-счётчиков")

# --- Шаг 2: пробуем получить историю по конкретному счётчику без доп. параметров ---
test_id = "MMM_EcoCompt_XTH19101158"  # подставь реальный id из ответа шага 1, если формат другой
r2 = try_get(f"/ecocounter_timeseries/{test_id}/attrs/intensity",
             label=f"История intensity для {test_id} (без параметров)")

# --- Шаг 3: пробуем с параметрами дат — частый паттерн для FIWARE QuantumLeap-подобных API ---
r3 = try_get(f"/ecocounter_timeseries/{test_id}/attrs/intensity",
             params={"fromDate": "2023-01-01T00:00:00", "toDate": "2023-01-08T00:00:00"},
             label=f"История intensity для {test_id} (с fromDate/toDate)")


=== Список эко-счётчиков ===
URL: https://portail-api-data.montpellier3m.fr/ecocounter params: None
Status: 502
Тело (первые 1500 симв.):
{
  "message":"An invalid response was received from the upstream server"
}

=== История intensity для MMM_EcoCompt_XTH19101158 (без параметров) ===
URL: https://portail-api-data.montpellier3m.fr/ecocounter_timeseries/MMM_EcoCompt_XTH19101158/attrs/intensity params: None
Ошибка: HTTPSConnectionPool(host='portail-api-data.montpellier3m.fr', port=443): Read timed out. (read timeout=20)

=== История intensity для MMM_EcoCompt_XTH19101158 (с fromDate/toDate) ===
URL: https://portail-api-data.montpellier3m.fr/ecocounter_timeseries/MMM_EcoCompt_XTH19101158/attrs/intensity params: {'fromDate': '2023-01-01T00:00:00', 'toDate': '2023-01-08T00:00:00'}
Ошибка: HTTPSConnectionPool(host='portail-api-data.montpellier3m.fr', port=443): Read timed out. (read timeout=20)


In [ ]:
# в Colab — идентичный запрос, без VPN, с другого IP
import requests

r = requests.get(
    "https://portail-api-data.montpellier3m.fr/ecocounter",
    params={"limit": 1000},
    headers={"accept": "application/json"},
    timeout=20,
)
print("Status:", r.status_code)
print(r.text[:500])

Status: 502
{
  "message":"An invalid response was received from the upstream server"
}


In [ ]:
r2 = requests.get(
    "https://portail-api-data.montpellier3m.fr/parkingspaces",
    params={"limit": 1000},
    headers={"accept": "application/json"},
    timeout=20,
)
print("Status:", r2.status_code)
print(r2.text[:500])

Status: 502
{
  "message":"An invalid response was received from the upstream server"
}
